In [ ]:
#Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

uploaded = files.upload()


Saving Ecommerce final dataset.xlsx to Ecommerce final dataset (1).xlsx


In [ ]:
file_name = next(iter(uploaded))
print("Dataset loaded successfully")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully


NameError: name 'df' is not defined

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
missing = (df.isnull().sum() / len(df)) * 100

missing.sort_values(ascending=False)

In [ ]:
df.drop(columns=[
    'review_comment_title',
    'review_comment_message'
], inplace=True)

In [ ]:
df.drop(columns=['customer_unique_id'], inplace=True)

In [ ]:
df.drop(columns=['seller_zip_code_prefix'], inplace=True)
df.drop(columns=['customer_zip_code_prefix'], inplace=True)

In [ ]:
df.shape

In [ ]:
print(df.columns)
print("Total Columns:", len(df.columns))

In [ ]:
#Filling Catogorical columns
df['product_category_name'] = df['product_category_name'].fillna('Unknown')

df['product_category_name_english'] = df['product_category_name_english'].fillna('Unknown')

In [ ]:
#Filling Numerical columns
product_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'product_description_lenght',
    'product_name_lenght',
    'product_photos_qty'
]

for col in product_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
#Seller Information
seller_cols = [
    'seller_id',
    'seller_city',
    'seller_state',

]

for col in seller_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
#Price and Freight
df['price'] = df['price'].fillna(df['price'].median())

df['freight_value'] = df['freight_value'].fillna(df['freight_value'].median())

In [ ]:
#Payment Columns
payment_cols = [
    'payment_type',
    'payment_sequential',
    'payment_installments',
    'payment_value'
]

for col in payment_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
df['review_score'] = df['review_score'].fillna(df['review_score'].median())

In [ ]:
# Handling order_delivered_customer_dateorder_delivered_carrier_date order_approved_at based on order_status
df['order_approved_at'] = df['order_approved_at'].fillna(df['order_purchase_timestamp'])

In [ ]:
df = df.dropna(subset=['order_item_id', 'product_id'])

In [ ]:
df.isnull().sum()

In [ ]:
#Validate Numerical Columns
df[['price',
    'freight_value',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm']].describe()

In [ ]:
#validating numerical columns
print("Price <= 0:", (df['price'] <= 0).sum())
print("Freight Value < 0:", (df['freight_value'] < 0).sum())
print("Product Weight <= 0:", (df['product_weight_g'] <= 0).sum())

In [ ]:
median_weight = df.loc[df['product_weight_g'] > 0, 'product_weight_g'].median()

df.loc[df['product_weight_g'] <= 0, 'product_weight_g'] = median_weight

In [ ]:
#Outlier Detection
numeric_cols = [
    'price',
    'freight_value',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

plt.figure(figsize=(15, 8))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[col])
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
#Handling Outliers
numeric_cols = [
    'price',
    'freight_value',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}: {len(outliers)} outliers")

In [ ]:
numeric_cols = [
    'price',
    'freight_value',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

df_clean = df.copy()

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    df_clean = df_clean[
        (df_clean[col] >= lower_limit) &
        (df_clean[col] <= upper_limit)
    ]

print("Original Shape:", df.shape)
print("New Shape:", df_clean.shape)

In [ ]:
df = df_clean


In [ ]:
df.to_csv("Supply_Chain_Cleaned_Dataset.csv", index=False)

In [ ]:
from google.colab import files
files.download("Supply_Chain_Cleaned_Dataset.csv")


In [ ]:
#Order Status Distribution (Bar Chart)
import matplotlib.pyplot as plt

df['order_status'].value_counts().plot(kind='bar', figsize=(8,5))

plt.title('Distribution of Order Status')
plt.xlabel('Order Status')
plt.ylabel('Number of Orders')
plt.show()

In [ ]:
#Top 10 Product Categories
import matplotlib.pyplot as plt

top10 = df['product_category_name'].value_counts().head(10)

plt.figure(figsize=(12,6))
top10.plot(kind='bar')

plt.title('Top 10 Product Categories')
plt.xlabel('Product Category')
plt.ylabel('Number of Products')
plt.xticks(rotation=45)

plt.show()

In [ ]:
#Price VS Freight value(transportation cost)
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.scatter(
    df['price'],
    df['freight_value'],
    s=8,          # Marker size
    alpha=0.4     # Transparency
)

plt.title('Relationship between Product Price and Freight Value')
plt.xlabel('Product Price')
plt.ylabel('Freight Value')

plt.grid(True)
plt.show()

In [ ]:
#Delivery Delay vs Review Score
#creating delivary delay column
df['delivery_delay'] = (
    df['order_delivered_customer_date'] -
    df['order_estimated_delivery_date']
).dt.days

In [ ]:
#delivary delay vs review score
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

df.boxplot(column='delivery_delay', by='review_score')

plt.title('Delivery Delay vs Review Score')
plt.suptitle('')  # Removes the default title
plt.xlabel('Review Score')
plt.ylabel('Delivery Delay (Days)')

plt.show()

In [ ]:
#Payment distribution
import matplotlib.pyplot as plt

payment_counts = df['payment_type'].value_counts()

plt.figure(figsize=(8,8))

plt.pie(
    payment_counts,
    autopct='%1.1f%%',
    startangle=90
)

plt.legend(
    payment_counts.index,
    title="Payment Type",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

plt.title("Payment Type Distribution")

plt.show()



In [ ]:
#Shows where most sellers are located
import matplotlib.pyplot as plt

top5 = df['seller_state'].value_counts().head(5)

plt.figure(figsize=(8,8))

top5.plot(
    kind='pie',
    autopct='%1.1f%%',
    startangle=90
)

plt.title('Top 5 Seller States')
plt.ylabel('')

plt.show()